In [39]:
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [21]:
df = pd.read_json(r"D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\web scraping\extraction\car_dataset_ahmedabad.json")

col_list = df.columns
col_list

Index(['url', 'car_name', 'Price', 'Registration Year', 'Insurance',
       'Fuel Type', 'Seats', 'Kms Driven', 'RTO', 'Ownership',
       ...
       'Secondary Fuel Type', 'Drag Coefficient',
       'Boot Space Rear Seat Folding', 'Approach Angle', 'Break-over Angle',
       'Departure Angle', 'Petrol Mileage (ARAI)',
       'Petrol Fuel Tank Capacity (Litres)', 'Acceleration 0-100kmph',
       'CNG Highway Mileage'],
      dtype='object', length=113)

In [22]:
# with open(r'D:\IMP  ML  PROJECTS\CAR PRICE PREDICTION\web scraping\feature_engineering\features.txt', 'w') as f:
#     for item in col_list:
#         f.write(item + '\n')

In [23]:
duplicate_count = df.duplicated().sum()

In [24]:
print(df.shape)
print(duplicate_count)

(1074, 113)
0


In [25]:
req_col = ['Mileage', 'Engine', 'Kerb Weight', 'Fuel', 'Transmission Type', 'Power', 'No. of Cylinders', 'Registration Year']
df = df[req_col]

In [26]:
df.head()

,Mileage,Engine,Kerb Weight,Fuel,Transmission Type,Power,No. of Cylinders,Registration Year
0,18.9 kmpl,1197 cc,935 kg,Petrol,Manual,82 bhp,4.0,2015
1,19.81 kmpl,1086 cc,860 kg,Petrol,Manual,68.05 bhp,4.0,Apr 2015
2,15.6 kmpl,1196 cc,1090 kg,Petrol,Manual,70 bhp,4.0,Dec 2019
3,18.9 kmpl,1197 cc,1060 kg,Petrol,Manual,81.86 bhp,4.0,Jul 2017
4,25.44 kmpl,936 cc,1025 kg,Diesel,Manual,56.3 bhp,3.0,2015


In [27]:
# Missing percentage count
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing")

Mileage: 5.03% missing
Engine: 1.30% missing
Kerb Weight: 7.82% missing
Fuel: 11.55% missing
Transmission Type: 0.28% missing
Power: 2.89% missing
No. of Cylinders: 0.84% missing
Registration Year: 0.28% missing


In [28]:
df['Fuel'].value_counts()

Fuel
Petrol    761
Diesel    161
CNG        28
Name: count, dtype: int64

1. Mileage

In [29]:
df['Mileage'] = df['Mileage'].str.extract(r'(\d+\.?\d*)').astype(float)

2. Engine

In [30]:
df['Engine'] = df['Engine'].str.extract(r'(\d+)').astype(float)

3. Weight

In [31]:
df['Kerb Weight'] = df['Kerb Weight'].str.extract(r'(\d+)').astype(float)

4. Power

In [32]:
df['Power'] = df['Power'].str.extract(r'(\d+\.?\d*)').astype(float)

5. Registration Year

In [33]:
df['Registration Year'] = pd.to_numeric(
    df['Registration Year'].str.extract(r'(\d{4})')[0],
    errors='coerce'
)

6. Transmission Type

In [34]:
df['Transmission Type'] = df['Transmission Type'].map({
                                'Automatic': 1,
                                'Manual': 0
                            })

---

In [35]:
num_cols = ['Mileage','Engine','Kerb Weight','Power','No. of Cylinders']

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

cat_cols = ['Fuel','Transmission Type','Registration Year']

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13444\584042974.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_13444\584042974.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

In [36]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [37]:
# features and target
X = df.drop(columns=['Mileage'])   
y = df['Mileage']

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# column to encode
categorical_features = ['Fuel']

# transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('fuel_encoder', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

# pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor())
])


In [38]:
# hyperparameter grid
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5]
}

# grid search
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# train
grid_search.fit(X_train, y_train)

# best model
best_model = grid_search.best_estimator_

# prediction
y_pred = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)
print("R2 Score:", grid_search.best_score_)

Best Parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 100}
R2 Score: 0.788624104414027
